# M32 — Control Inference and Adaptation

**Objective:** manipulate inference behavior and understand adaptation options.

M31 already produced a language-model identity. The useful whole here is
**token selection plus the adaptation choice**, not another training run:

`checkpoint → prompt/context → logits → temperature → top-k/top-p → greedy or sample → stop → tokens + InferenceConfig`

Temperature and filters change a **distribution over already-scored tokens**.
They are not quality knobs. Search, RAG, and tool execution stay closed.
The 4-token fixture is **not a production** decoder.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a greedy index, a probability table, a stop
reason, a mismatched config field, or an adaptation route.

Do not treat temperature as quality. Do not fine-tune first for a freshness
problem. The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M32" / "inference_adaptation.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M31.llm_training_core import (
    TRAINING_TIME_BOUNDARY,
    TRAINING_VERSION,
    StageAwareCheckpoint,
)
from missions.M32.inference_adaptation import (
    ADAPTATION_HIERARCHY,
    FILTER_LOGITS,
    GREEDY_LOGITS,
    INFERENCE_VERSION,
    LiveAdapterUnavailable,
    PROMPT_A,
    PROMPT_B,
    SCALE_LIMIT,
    SEED,
    SEED_OTHER,
    SYSTEM_MAP,
    TEMP_LOGITS,
    TEMPERATURES,
    TIE_LOGITS,
    VOCAB,
    apply_top_k,
    apply_top_p,
    attach_m31_checkpoint,
    compare_outputs,
    compare_outputs_naive,
    config_as_evidence,
    decide_scenario,
    entropy,
    first_divergence,
    greedy_token,
    make_config,
    observability_report,
    optional_live_complete,
    pipeline_with_defect,
    prepare_distribution,
    repair_run,
    run_inference,
    sample_once,
    softmax_probs,
    token_names,
    training_time_boundary,
)

print("repository root:", ROOT)
print("inference version:", INFERENCE_VERSION)
print("training version:", TRAINING_VERSION)
print("vocab:", VOCAB)
print("scale:\n", SCALE_LIMIT)
print("boundary:\n", TRAINING_TIME_BOUNDARY)


## M31 → M32 boundary: the checkpoint in, token selection out

M31 handed a `StageAwareCheckpoint` with `training_time=False` and
`inference_ready=True`. Training time updates weights under a declared
objective. Inference time **consumes** that checkpoint; weights stay frozen.

M32 does **not** retrain the table. Teaching scale is a 4-token local
score fixture plus an explicit `InferenceConfig`. How tokens are selected
from logits is this mission. How documents are retrieved, packed, or
tooled is not.

Canonical sources: `hf-llm-course` and `karpathy-zero-to-hero`.


## Frozen teaching fixtures

Declare the useful whole **before** the first argmax.

| Knob | Teaching value |
| --- | --- |
| Decoder | local prefix → logits table (`alpha` `beta` `gamma` `stop`) |
| Checkpoint | attached M31 identity `v07-teaching-lm-1` |
| Greedy logits | `(1.0, 3.0, 2.0, 0.0)` |
| Temperature logits | `(log 3, 0)` |
| Filter counts | `(10, 6, 3, 1)` |
| Seed / other seed | `3201` / `3203` |
| Filter order | temperature → top-k → top-p → softmax |
| Live adapter | optional, fail-closed, not required |

The table is authored so sampling bugs are visible. It is **not a production**
decoder and not a quality benchmark.


In [ ]:
checkpoint = attach_m31_checkpoint()
print("checkpoint type", type(checkpoint).__name__)
print("checkpoint id", checkpoint.checkpoint_id)
print("version", checkpoint.version, "stage", checkpoint.adaptation_stage)
print("training_time", checkpoint.training_time, "inference_ready", checkpoint.inference_ready)
print("isinstance StageAwareCheckpoint", isinstance(checkpoint, StageAwareCheckpoint))
print("greedy logits", GREEDY_LOGITS)
print("temp logits", TEMP_LOGITS)
print("filter logits", FILTER_LOGITS)
print("prompt A", PROMPT_A, "prompt B", PROMPT_B)
print("seed", SEED, "other", SEED_OTHER)
print("M31 boundary matches", training_time_boundary() == TRAINING_TIME_BOUNDARY)
assert checkpoint.inference_ready and not checkpoint.training_time
print("produced LM identity attached; decoder is still a local table")


The attached checkpoint is lineage and stage, not a hidden model download.
`training_time` is already false: this mission is not allowed to pretend a
sampling change is a training-stage change. The vocab has four teaching
tokens so every probability is hand-checkable.


## Predict before running — whole inference system

Timestamp a prediction before `run-whole`.

The teaching system map prints, then the attached checkpoint is checked
and the optional live adapter is shown to fail closed.

Predict:
- whether this cell updates M31 weights
- whether a live model is required for the canonical path
- which side of the **training-time** / inference-time boundary `run_inference` sits on


In [ ]:
print(SYSTEM_MAP)
print("--- hierarchy ---")
print(ADAPTATION_HIERARCHY)
print("attached checkpoint", checkpoint.checkpoint_id)
print("boundary:", training_time_boundary())
try:
    optional_live_complete("hello", make_config())
    live_status = "unexpected-success"
except LiveAdapterUnavailable as exc:
    live_status = str(exc)
    print("live adapter fail-closed:", exc)
assert checkpoint.training_time is False
assert "not required" in live_status.lower() or "local logits" in live_status.lower()
print("weights stay frozen; token selection is a separate control surface")


The map ends at tokens plus `InferenceConfig`. Retrieval and tools appear
only as **named later levers**. The live adapter raising is evidence that
the canonical path is the local table, not a paid API.


## Predict before running — greedy selection

Timestamp a prediction before `run-greedy`.

Fixed logits `(1.0, 3.0, 2.0, 0.0)` and a tie `(2.0, 2.0, 1.0, 0.0)`.

Predict:
- the greedy index on the first vector
- which index wins the tie, and why
- whether dividing every logit by 2 would change the greedy index


In [ ]:
greedy_idx = greedy_token(GREEDY_LOGITS)
tie_idx = greedy_token(TIE_LOGITS)
scaled_idx = greedy_token(tuple(value / 2.0 for value in GREEDY_LOGITS))
print("vocab", VOCAB)
print("greedy logits", GREEDY_LOGITS, "->", greedy_idx, VOCAB[greedy_idx])
print("tie logits", TIE_LOGITS, "->", tie_idx, VOCAB[tie_idx])
print("scaled greedy index", scaled_idx)
assert greedy_idx == 1
assert tie_idx == 0
assert scaled_idx == greedy_idx
print("greedy is argmax; lowest index wins ties; T>0 scaling does not move argmax")


Greedy decoding is deterministic given the logits. A temperature greater
than zero rescales the vector but **does not change argmax**. If you want
randomness you must sample, and then you must record a seed.


## Predict before running — temperature

Timestamp a prediction before `run-temperature`.

Same two-class logits `(log 3, 0)`. Change only temperature
(`0.5`, `1.0`, `2.0`).

Predict:
- the `T=1` probabilities (hand-computable)
- whether `T=0.5` concentrates or flattens the mass
- whether the greedy index moves


In [ ]:
temp_rows = []
for temperature in TEMPERATURES:
    probs = softmax_probs(TEMP_LOGITS, temperature)
    temp_rows.append((temperature, probs, entropy(probs), greedy_token(TEMP_LOGITS)))
    print(f"T={temperature} probs={tuple(round(p, 6) for p in probs)} H={entropy(probs):.6f}")
p_half, p_one, p_two = (row[1] for row in temp_rows)
assert abs(p_one[0] - 0.75) < 1e-12
assert abs(p_half[0] - 0.9) < 1e-12
assert abs(p_two[0] - math.sqrt(3.0) / (math.sqrt(3.0) + 1.0)) < 1e-12
assert entropy(p_half) < entropy(p_one) < entropy(p_two)
assert len({row[3] for row in temp_rows}) == 1
print("greedy index stayed", temp_rows[0][3], "across temperatures")
print("temperature is a distribution knob, not a quality knob")


At `T=1`, `(log 3, 0)` is exactly `(0.75, 0.25)`. Lower `T` concentrates
mass; higher `T` raises entropy toward uniform. None of those statements
is "the answer got better." Quality is a later evaluation mission.


## Predict before running — top-k / top-p

Timestamp a prediction before `run-filters`.

Fixed logits from counts `(10, 6, 3, 1)` and temperature `1`. First
apply `top-k=2`. Then, separately, apply `top-p=0.50` and `top-p=0.81`.

Predict:
- which classes survive `top-k=2` and the renormalized masses
- whether `top-p=0.50` keeps one class or two
- whether `top-k=2` and `top-p=0.81` agree


In [ ]:
base = softmax_probs(FILTER_LOGITS, 1.0)
print("T=1 probs", tuple(round(p, 6) for p in base))
masked_k = apply_top_k(FILTER_LOGITS, 2)
_, topk_probs = prepare_distribution(FILTER_LOGITS, temperature=1.0, top_k=2)
_, topp_one = prepare_distribution(FILTER_LOGITS, temperature=1.0, top_p=0.50)
_, topp_three = prepare_distribution(FILTER_LOGITS, temperature=1.0, top_p=0.81)
print("top-k=2 masked", masked_k)
print("top-k=2 probs", tuple(round(p, 6) for p in topk_probs))
print("top-p=0.50 probs", tuple(round(p, 6) for p in topp_one))
print("top-p=0.81 probs", tuple(round(p, 6) for p in topp_three))
print("apply_top_p name in order after", apply_top_p.__name__)
assert abs(base[0] - 0.5) < 1e-12
assert abs(topk_probs[0] - 0.625) < 1e-12
assert abs(topp_one[0] - 1.0) < 1e-12
assert topp_three[2] > 0.0 and topk_probs[2] == 0.0
print("filters truncate then renormalize; they are not quality knobs")


`top-k` keeps a count of leading logits. `top-p` keeps a probability
budget. On this fixture they can agree (`k=2` vs a nucleus that also
keeps two) or disagree (`p=0.81` keeps a third class). Teaching order is
temperature, then top-k, then top-p, then softmax.


## Predict before running — seed replay

Timestamp a prediction before `run-seed`.

Sample once from the filter logits at `T=1` with no extra filters.
Repeat seed `3201`, then use a different seed. Other settings stay fixed.

Predict:
- whether the first seed replays
- whether the second seed is allowed to differ
- what you would still have to log besides the seed


In [ ]:
sample_a = sample_once(FILTER_LOGITS, seed=SEED)
sample_replay = sample_once(FILTER_LOGITS, seed=SEED)
sample_b = sample_once(FILTER_LOGITS, seed=SEED_OTHER)
print("seed", SEED, "token", sample_a.token_id, VOCAB[sample_a.token_id])
print("replay", sample_replay.token_id)
print("other seed", SEED_OTHER, "token", sample_b.token_id, VOCAB[sample_b.token_id])
assert sample_a.token_id == sample_replay.token_id
assert sample_a.token_id != sample_b.token_id
print("replay is a sampling claim; it is not proof the checkpoint changed")


Same logits, same temperature, same seed → same draw. A different seed
may emit a different token without anyone training a new model. If the
seed is missing from the log, you cannot tell those stories apart.


## Predict before running — inference config as evidence

Timestamp a prediction before `run-config`.

Build one `InferenceConfig` on the attached checkpoint and print the
evidence record.

Predict:
- whether `training_time` is allowed to be true
- which fields a later mission needs to replay this call
- whether the fingerprint should change if only temperature changes


In [ ]:
cfg = make_config(prompt_ids=PROMPT_A, temperature=1.0, seed=SEED, do_sample=True, max_tokens=4)
evidence = config_as_evidence(cfg)
for key, value in evidence.items():
    print(f"{key}: {value}")
cfg_hot = make_config(prompt_ids=PROMPT_A, temperature=1.7, seed=SEED, do_sample=True, max_tokens=4)
print("fingerprint moved with temperature", evidence["fingerprint"] != config_as_evidence(cfg_hot)["fingerprint"])
assert evidence["training_time"] is False
assert evidence["checkpoint_id"] == checkpoint.checkpoint_id
print("config is reproducibility evidence, not a leaderboard score")


An output string without checkpoint id, prompt, temperature, seed, stop,
and max-tokens is not a reproducible inference result. Later missions
inherit this provider log, not a mystery `generate()` call.


## Predict before running — stop / max-tokens

Timestamp a prediction before `run-stop`.

Greedy generation from prompt A. Change **only** `max_tokens`
(`8` versus `2`). Stop token stays `stop`. Sampling stays off.

Predict:
- the greedy continuation when the budget is large enough to hit `stop`
- the continuation and `stop_reason` when the budget is `2`


In [ ]:
cfg_long = make_config(prompt_ids=PROMPT_A, do_sample=False, max_tokens=8)
cfg_short = make_config(prompt_ids=PROMPT_A, do_sample=False, max_tokens=2)
gen_long = run_inference(PROMPT_A, cfg_long)
gen_short = run_inference(PROMPT_A, cfg_short)
print("long", gen_long.generated_ids, token_names(gen_long.generated_ids), gen_long.stop_reason)
print("short", gen_short.generated_ids, token_names(gen_short.generated_ids), gen_short.stop_reason)
print("divergence", first_divergence(gen_long, gen_short))
assert gen_long.generated_ids == (1, 2, 3)
assert gen_long.stop_reason == "stop_token"
assert gen_short.generated_ids == (1, 2)
assert gen_short.stop_reason == "max_tokens"
assert first_divergence(gen_long, gen_short) == "max_tokens"
print("termination changed; the checkpoint did not")


A truncated string is not automatically a worse model. It may be a
smaller budget. Stop conditions belong in the same evidence record as
temperature and seed.


## Predict before running — prompt / context

Timestamp a prediction before `run-prompt`.

Hold sampling fixed (greedy, same seed, same stop, same checkpoint).
Change **only** the prompt prefix: `PROMPT_A=(0,)` versus `PROMPT_B=(1,)`.

Predict:
- whether the two greedy continuations match
- whether a continuation change here is evidence of a new checkpoint


In [ ]:
cfg_a = make_config(prompt_ids=PROMPT_A, do_sample=False)
cfg_b = make_config(prompt_ids=PROMPT_B, do_sample=False)
gen_a = run_inference(PROMPT_A, cfg_a)
gen_b = run_inference(PROMPT_B, cfg_b)
print("prompt A", gen_a.prompt_ids, "->", token_names(gen_a.generated_ids), gen_a.stop_reason)
print("prompt B", gen_b.prompt_ids, "->", token_names(gen_b.generated_ids), gen_b.stop_reason)
print("same checkpoint", gen_a.checkpoint_id == gen_b.checkpoint_id)
print("same seed", gen_a.config.seed == gen_b.config.seed)
print("divergence", first_divergence(gen_a, gen_b))
assert gen_a.generated_ids != gen_b.generated_ids
assert first_divergence(gen_a, gen_b) == "prompt_ids"
assert gen_a.config.temperature == gen_b.config.temperature
print("context selected a different score row; weights did not move")


Prompt/context is the cheapest adaptation lever: change what a frozen
checkpoint sees. It is not retrieval (no corpus lookup), not a tool
call, and not a weight update. Those distinctions are the rest of V07.


## Predict before running — adaptation rubric

Timestamp a prediction before `run-adaptation`.

Four problems, one frozen rubric: casual tone; private vendor PDF;
exact VAT on an invoice; a product that must refuse competitor praise
on every surface.

Predict one route each: prompt, retrieval, tools, or parameters.
Do not implement a search service or a tool executor.


In [ ]:
adaptation_cases = (
    "email_tone",
    "vendor_policy",
    "invoice_vat",
    "always_refuse_competitor_praise",
)
decisions = []
for case_id in adaptation_cases:
    decision = decide_scenario(case_id)
    decisions.append(decision)
    print(case_id, "->", decision.chosen_route)
    print(" ", decision.problem)
    print(" ", decision.signals)
assert [d.chosen_route for d in decisions] == ["prompt", "retrieval", "tools", "parameters"]
print("--- hierarchy ---")
print(ADAPTATION_HIERARCHY)
print("fine-tuning is last, not first")


The rubric is a teaching default, not your ADR. Fresh or private facts
go to retrieval (M33/M34 will implement lookup). Arithmetic goes to
tools (M37 will execute). Persistent behavior that prompt cannot hold
may justify instruction-tuning or LoRA — named here, not trained here.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(VOCAB))
width = 0.25
for offset, temperature in zip((-width, 0.0, width), TEMPERATURES):
    probs = softmax_probs(FILTER_LOGITS, temperature)
    ax.bar(x + offset, probs, width=width, label=f"T={temperature}")
ax.set_xticks(x)
ax.set_xticklabels(VOCAB)
ax.set_ylabel("softmax probability")
ax.set_xlabel("token")
ax.set_title("Same logits, different temperature (not a quality ranking)")
ax.legend()
ax.set_ylim(0, 1)
fig.tight_layout()
plt.show()
print("plot asks: did mass concentrate as T fell? it does not rank answers")


## Predict before running — code reading

Timestamp a prediction before `run-code-reading`.

Read `run_inference`, `prepare_distribution`, `repair_run`, and
`optional_live_complete` in `missions/M32/inference_adaptation.py`.

Predict:
- whether greedy selection depends on temperature for `T > 0`
- which metadata field would have to match before two outputs can be
  compared as a model change
- what `optional_live_complete` does on the canonical path

Do not search the file for a vector index or a tool executor.


In [ ]:
infer_src = inspect.getsource(run_inference)
prep_src = inspect.getsource(prepare_distribution)
repair_src = inspect.getsource(repair_run)
live_src = inspect.getsource(optional_live_complete)
print("prepare_distribution\n", prep_src)
print("--- run_inference (config, stop, fallback) ---")
for line in infer_src.splitlines():
    if any(token in line for token in ("training_time", "stop", "do_sample", "seed", "local_logits", "Live")):
        print(line)
print("--- repair_run ---")
print(repair_src)
print("--- optional_live_complete ---")
print(live_src)
report = observability_report(gen_a)
print("handoff", report["handoff"])
assert "temperature scale" in prep_src
assert "top-k" in prep_src
assert "reference_config" in repair_src
assert "LiveAdapterUnavailable" in live_src
print("prompt, scores, filters, sample or greedy, stop, metadata, fail-closed live adapter")


## Predict before running — Controlled failure: two completions

Timestamp a prediction before `run-failure-uncontrolled`.

Two sampled completions share a checkpoint identity. A naive compare
treats token disagreement as a model change.

Predict:
- which recorded fields would have to match before that claim is legal
- whether you should reach for a new checkpoint first


In [ ]:
broken_settings = pipeline_with_defect(defect="uncontrolled_settings")
print("defect", broken_settings.defect, "claim", broken_settings.claim)
print("mismatched", broken_settings.mismatched_fields)
print("naive", broken_settings.audit["naive_compare"])
print("controlled", broken_settings.audit["controlled_compare"])
print("left T,seed", broken_settings.left.config.temperature, broken_settings.left.config.seed, token_names(broken_settings.left.generated_ids))
print("right T,seed", broken_settings.right.config.temperature, broken_settings.right.config.seed, token_names(broken_settings.right.generated_ids))
print("same checkpoint", broken_settings.left.checkpoint_id == broken_settings.right.checkpoint_id)
print("first divergence", first_divergence(broken_settings.left, broken_settings.right))
print("naive fn", compare_outputs_naive(broken_settings.left, broken_settings.right))
print("controlled fn", compare_outputs(broken_settings.left, broken_settings.right))
assert broken_settings.claim == "model_changed"
assert first_divergence(broken_settings.left, broken_settings.right) == "temperature"
print("token strings differed; the checkpoint id did not")


## Predict before running — Controlled failure: stale hours

Timestamp a prediction before `run-failure-adaptation`.

A team proposes changing weights because answers still quote last year's
holiday hours for a site.

Predict:
- whether that is the smallest lever
- which signal (freshness, computation, style, weight change) is actually on


In [ ]:
broken_adapt = pipeline_with_defect(defect="wrong_adaptation")
print("defect", broken_adapt.defect, "claim", broken_adapt.claim)
print("proposed route", broken_adapt.decision.chosen_route)
print("problem", broken_adapt.decision.problem)
print("signals", broken_adapt.decision.signals)
print("rationale", broken_adapt.decision.rationale)
assert broken_adapt.decision.chosen_route == "parameters"
assert broken_adapt.decision.signals["freshness"]
print("weights were proposed; check whether facts went stale instead")


## Diagnosis record (your log, not this repo)

Fill symptom, plausible hypotheses, discriminating experiment, observed
result, root cause, smallest repair, verification, and regression
evidence in your evidence log.

Discriminators live in `InferenceConfig` fields and adaptation signals.
Do not start with a new model or a lower loss.


## Predict before running — repair from the broken traces

Timestamp a prediction before `run-failure-repair`.

`repair_run` must reuse each broken trace's reference config or signals,
with the named defect cleared.

Predict:
- whether repaired samples match when controls are held
- whether the broken objects still diverge after repair
- whether the stale-hours case still proposes a weight update after repair


In [ ]:
repaired_settings = repair_run(broken_settings)
repaired_adapt = repair_run(broken_adapt)
print("repaired settings defect", repaired_settings.defect, "claim", repaired_settings.claim)
print("repaired tokens", token_names(repaired_settings.left.generated_ids), token_names(repaired_settings.right.generated_ids))
print("reused seed", repaired_settings.left.config.seed, "T", repaired_settings.left.config.temperature)
print("broken still diverges", first_divergence(broken_settings.left, broken_settings.right))
print("repaired adapt route", repaired_adapt.decision.chosen_route)
print("broken adapt still", broken_adapt.decision.chosen_route)
assert repaired_settings.left.generated_ids == repaired_settings.right.generated_ids
assert repaired_settings.left.config == broken_settings.reference_config
assert first_divergence(broken_settings.left, broken_settings.right) == "temperature"
assert repaired_adapt.decision.chosen_route != broken_adapt.decision.chosen_route
assert repaired_adapt.decision.signals == broken_adapt.decision.signals
print("repair restored controls / lever; defective objects remain as regression fixtures")


Repair did not invent a second unrelated happy-path run from scratch
and did not open a search index. It reused the broken object's
checkpoint and signals. The defective traces still disagree: that is
the regression.


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- greedy index and tie-break on the teaching logits
- temperature table with entropy, greedy index unchanged
- top-k / top-p truncation table
- seed replay pair
- stop-reason pair
- two-prompt continuations with sampling held
- adaptation routes for the four rubric cases
- uncontrolled-settings and wrong-lever diagnosis plus `repair_run`

See `missions/M32/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M32/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

Compute greedy and softmax on a fresh three-class vector, list missing
run metadata, choose an adaptation lever for a cafeteria-menu complaint,
and state what must be logged to replay a sampled completion.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M32/adr_prompt.md` to choose a V07 adaptation hierarchy
and inference-configuration policy (prompt / retrieval / tools /
parameters, required logs, revisit triggers). Do not claim a production
decoder and do not implement search, RAG, or tools here.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR. Formal engineering review is required
at M32 for the package, not as a substitute for the learner ADR.


## M32 → M33 / M34 / M37 handoff

M31 supplied a checkpoint. M32 attached token selection, an
`InferenceConfig` provider log, and an adaptation decision hierarchy.

M33 may build semantic search. M34 may ground generation. M37 may
execute tools. They must consume this configuration contract. They
must not relabel a sampling change as a training-stage change, and they
must not fine-tune first for a freshness problem.

Reusable artifacts: `InferenceConfig`, `config_fingerprint`, attached
`v07-teaching-lm-1` identity, version `v07-teaching-inference-1`, and
the hierarchy sentence.


## Mission summary prompt

In your own words, using only observations from this lab:

1. Why is greedy invariant to temperature for `T > 0`?
2. What identity tells you two completions are comparable as a model change?
3. Why is a different sampled string insufficient evidence of a new checkpoint?
4. When is fine-tuning the wrong first response?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert greedy_idx == 1 and tie_idx == 0
assert abs(p_one[0] - 0.75) < 1e-12
assert abs(p_half[0] - 0.9) < 1e-12
assert abs(topk_probs[0] - 0.625) < 1e-12
assert sample_a.token_id == sample_replay.token_id
assert sample_a.token_id != sample_b.token_id
assert gen_long.stop_reason == "stop_token" and gen_short.stop_reason == "max_tokens"
assert first_divergence(gen_a, gen_b) == "prompt_ids"
assert [d.chosen_route for d in decisions] == ["prompt", "retrieval", "tools", "parameters"]
assert first_divergence(broken_settings.left, broken_settings.right) == "temperature"
assert compare_outputs_naive(broken_settings.left, broken_settings.right) == "model_changed"
assert repaired_settings.left.config == broken_settings.reference_config
assert broken_adapt.decision.chosen_route == "parameters"
assert repaired_adapt.decision.chosen_route == "retrieval"
assert checkpoint.inference_ready and not checkpoint.training_time
print("M32 integrity checks passed")
